# Comparative Analysis of Neural Machine Translation (NMT) Architectures
**Course Assignment:** Neural Machine Translation Comparative Study  
**Roll Number:** 6 (`random_state = 6`)  
**Dataset:** ManyThings.org Anki Language Pair

---  
### Models Evaluated:
1. **Recurrent Neural Network (Vanilla RNN)**
2. **Encoder–Decoder Architecture (Seq2Seq)**
3. **Encoder–Decoder with Additive (Bahdanau) Attention**
4. **Encoder–Decoder with Multiplicative (Luong) Attention**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
import numpy as np
import re
import math
import time
import matplotlib.pyplot as plt
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ------------------------------------------------------------------
# Reproducibility Setup (Roll Number: 6)
# ------------------------------------------------------------------
SEED = 6
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[System] Operating Device: {device} | Random Seed: {SEED}")

## 1. Preprocessing & Dataset Pipeline

In [ ]:
class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.n_words = 4

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2idx:
            self.word2idx[word] = self.n_words
            self.idx2word[self.n_words] = word
            self.n_words += 1

def normalize_string(s):
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s.strip()

def load_and_preprocess(pairs_data, max_len=12):
    src_vocab = Vocabulary("source")
    tgt_vocab = Vocabulary("target")
    cleaned_pairs = []

    for src, tgt in pairs_data:
        src_norm = normalize_string(src)
        tgt_norm = normalize_string(tgt)
        if len(src_norm.split()) <= max_len and len(tgt_norm.split()) <= max_len:
            src_vocab.add_sentence(src_norm)
            tgt_vocab.add_sentence(tgt_norm)
            cleaned_pairs.append((src_norm, tgt_norm))

    return src_vocab, tgt_vocab, cleaned_pairs

# Sample Bilingual Dataset Construction (e.g., English -> French format)
raw_dataset = [
    ("go .", "va !"),
    ("hi .", "salut !"),
    ("run .", "cours !"),
    ("i run .", "je cours ."),
    ("who is he ?", "qui est il ?"),
    ("i am a student .", "je suis etudiant ."),
    ("she likes dark coffee .", "elle aime le cafe noir ."),
    ("we are learning artificial intelligence .", "nous apprenons intelligence artificielle .")
] * 100

src_vocab, tgt_vocab, pairs = load_and_preprocess(raw_dataset)

# Train-Test Split using seed = 6
random.seed(SEED)
random.shuffle(pairs)
split_ratio = 0.8
split_idx = int(len(pairs) * split_ratio)
train_pairs = pairs[:split_idx]
test_pairs = pairs[split_idx:]

print(f"Total Cleaned Pairs: {len(pairs)}")
print(f"Train Pairs: {len(train_pairs)} | Test Pairs: {len(test_pairs)}")
print(f"Source Vocab Size: {src_vocab.n_words} | Target Vocab Size: {tgt_vocab.n_words}")

## 2. NMT Architecture Definitions

### Models:
1. **Vanilla RNN Model**
2. **Seq2Seq Encoder-Decoder Model**
3. **Seq2Seq + Additive Attention (Bahdanau)**
4. **Seq2Seq + Multiplicative Attention (Luong)**

In [ ]:
# ------------------------------------------------------------------
# Model 1: Single Vanilla RNN Architecture
# ------------------------------------------------------------------
class VanillaRNNNMT(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, hidden_dim):
        super(VanillaRNNNMT, self).__init__()
        self.embedding = nn.Embedding(src_vocab_size, hidden_dim)
        self.rnn = nn.RNN(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tgt_vocab_size)

    def forward(self, src):
        embedded = self.embedding(src)
        out, _ = self.rnn(embedded)
        logits = self.fc(out)
        return logits

# ------------------------------------------------------------------
# Model 2: Standard Encoder-Decoder (Seq2Seq)
# ------------------------------------------------------------------
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(input_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.gru(embedded)
        return outputs, hidden

class Seq2SeqDecoder(nn.Module):
    def __init__(self, output_dim, hidden_dim):
        super(Seq2SeqDecoder, self).__init__()
        self.embedding = nn.Embedding(output_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, tgt_input, hidden):
        embedded = self.embedding(tgt_input)
        output, hidden = self.gru(embedded, hidden)
        logits = self.fc(output)
        return logits, hidden

# ------------------------------------------------------------------
# Model 3: Additive Attention Decoder (Bahdanau)
# ------------------------------------------------------------------
class BahdanauDecoder(nn.Module):
    def __init__(self, output_dim, hidden_dim):
        super(BahdanauDecoder, self).__init__()
        self.embedding = nn.Embedding(output_dim, hidden_dim)
        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)
        self.gru = nn.GRU(hidden_dim * 2, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, tgt_input, hidden, encoder_outputs):
        embedded = self.embedding(tgt_input)
        hidden_proj = self.W1(hidden.transpose(0, 1))
        encoder_proj = self.W2(encoder_outputs)
        score = torch.tanh(hidden_proj + encoder_proj)
        attn_weights = torch.softmax(self.V(score), dim=1)
        context = torch.bmm(attn_weights.transpose(1, 2), encoder_outputs)
        concat = torch.cat((embedded, context), dim=2)
        output, hidden = self.gru(concat, hidden)
        logits = self.fc(output)
        return logits, hidden, attn_weights

# ------------------------------------------------------------------
# Model 4: Multiplicative Attention Decoder (Luong)
# ------------------------------------------------------------------
class LuongDecoder(nn.Module):
    def __init__(self, output_dim, hidden_dim):
        super(LuongDecoder, self).__init__()
        self.embedding = nn.Embedding(output_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        self.Wa = nn.Linear(hidden_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, tgt_input, hidden, encoder_outputs):
        embedded = self.embedding(tgt_input)
        gru_out, hidden = self.gru(embedded, hidden)
        score = torch.bmm(self.Wa(encoder_outputs), gru_out.transpose(1, 2))
        attn_weights = torch.softmax(score, dim=1)
        context = torch.bmm(attn_weights.transpose(1, 2), encoder_outputs)
        concat = torch.cat((gru_out, context), dim=2)
        logits = self.fc(concat)
        return logits, hidden, attn_weights

## 3. Unified Helper Utilities & Evaluation Mechanics

In [ ]:
def sentence_to_tensor(vocab, sentence):
    indexes = [vocab.word2idx.get(w, vocab.word2idx["<UNK>"]) for w in sentence.split(' ')]
    indexes.append(vocab.word2idx["<EOS>"])
    return torch.tensor(indexes, dtype=torch.long, device=device).unsqueeze(0)

def calculate_bleu_score(candidate_str, reference_str):
    smooth = SmoothingFunction().method1
    reference = [reference_str.split(' ')]
    candidate = candidate_str.split(' ')
    return sentence_bleu(reference, candidate, smoothing_function=smooth)

## 4. Execution Sandbox & Comparative Experiment Training Loop

In [ ]:
HIDDEN_DIM = 64
EPOCHS = 10
LR = 0.001

# Instantiate Models
encoder = Encoder(src_vocab.n_words, HIDDEN_DIM).to(device)
seq2seq_decoder = Seq2SeqDecoder(tgt_vocab.n_words, HIDDEN_DIM).to(device)
bahdanau_decoder = BahdanauDecoder(tgt_vocab.n_words, HIDDEN_DIM).to(device)
luong_decoder = LuongDecoder(tgt_vocab.n_words, HIDDEN_DIM).to(device)

criterion = nn.CrossEntropyLoss()

def train_model(model_name, epochs=EPOCHS):
    print(f"\n--- Training Model: {model_name} ---")
    losses = []
    
    if model_name == "Seq2Seq":
        optimizer = optim.Adam(list(encoder.parameters()) + list(seq2seq_decoder.parameters()), lr=LR)
    elif model_name == "Additive_Attention":
        optimizer = optim.Adam(list(encoder.parameters()) + list(bahdanau_decoder.parameters()), lr=LR)
    elif model_name == "Multiplicative_Attention":
        optimizer = optim.Adam(list(encoder.parameters()) + list(luong_decoder.parameters()), lr=LR)
    
    for epoch in range(1, epochs + 1):
        epoch_loss = 0
        for src_str, tgt_str in train_pairs:
            src_tensor = sentence_to_tensor(src_vocab, src_str)
            tgt_tensor = sentence_to_tensor(tgt_vocab, tgt_str)
            
            optimizer.zero_grad()
            encoder_outputs, hidden = encoder(src_tensor)
            
            loss = 0
            tgt_len = tgt_tensor.size(1)
            decoder_input = torch.tensor([[tgt_vocab.word2idx["<SOS>"]]], device=device)
            
            for t in range(tgt_len):
                if model_name == "Seq2Seq":
                    out, hidden = seq2seq_decoder(decoder_input, hidden)
                elif model_name == "Additive_Attention":
                    out, hidden, _ = bahdanau_decoder(decoder_input, hidden, encoder_outputs)
                elif model_name == "Multiplicative_Attention":
                    out, hidden, _ = luong_decoder(decoder_input, hidden, encoder_outputs)
                
                loss += criterion(out.squeeze(1), tgt_tensor[:, t])
                decoder_input = tgt_tensor[:, t].unsqueeze(0)
            
            loss.backward()
            optimizer.step()
            epoch_loss += (loss.item() / tgt_len)
        
        avg_loss = epoch_loss / len(train_pairs)
        losses.append(avg_loss)
        if epoch % 2 == 0:
            print(f"Epoch {epoch}/{epochs} | Loss: {avg_loss:.4f}")
            
    return losses

# Run training across models
seq2seq_losses = train_model("Seq2Seq")
additive_losses = train_model("Additive_Attention")
multiplicative_losses = train_model("Multiplicative_Attention")

## 5. Visualizing Loss Curves & Evaluation Comparison

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(seq2seq_losses, label="Standard Seq2Seq", marker="o")
plt.plot(additive_losses, label="Additive (Bahdanau) Attention", marker="s")
plt.plot(multiplicative_losses, label="Multiplicative (Luong) Attention", marker="^")
plt.title("Training Loss Convergence Across NMT Architectures")
plt.xlabel("Epochs")
plt.ylabel("Cross Entropy Loss")
plt.legend()
plt.grid(True)
plt.show()